# Coherent MXene/metal interface builder, step by step

1. MXene: conventional (orthogonal) cell, vacuum only below the slab.
2. Metal: conventional cubic cell from its space group.
3. In-plane replicas of each that give (approximately) the same box. The metal is the substrate and keeps its
   lattice; the MXene takes the strain.
4. Metal (001) placed on MXene (0001), with the gap and in-plane shift below.

Lattice constants come from the spreadsheet. Ordered MXenes and the FCC and L1₂ metals can be used. Solid solutions, Tx and Ni3(Al,Ti) cannot.

In [1]:
import math
import re

import numpy as np
import openpyxl
from pymatgen.core import Lattice, Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

In [ ]:
SPREADSHEET = "../data/MXene_FCC_L12_strain_matrix.xlsx"

MXENE = "Ti3C2"        # MXenes sheet, Formula
STACKING = "ABC"       # ABC = AβC (octahedral), ABA = AβA (trigonal prismatic)
VACUUM = 20.0          # Å, below the MXene only

METAL = "Au"           # Metals sheet, Symbol
METAL_LAYERS = 4       # metal (001) atomic planes

GAP = 2.3              # Å, top MXene layer to first metal layer
SHIFT = (0.0, 0.0)     # Å, in-plane shift of the metal along x and y
TOL = 0.02             # largest |strain| allowed on the MXene along x and along y
MAX_REPLICAS = 10      # largest number of metal replicas tried per direction

## Lattice constants from the spreadsheet

In [ ]:
book = openpyxl.load_workbook(SPREADSHEET, read_only=True, data_only=True)
mxenes = {f: a for f, _, _, a, *_ in book["MXenes"].iter_rows(min_row=3, values_only=True)
          if f and isinstance(a, (int, float)) and re.fullmatch(r"([A-Z][a-z]?\d*)+", f) and "Tx" not in f}
metals = {s: (a, structure) for s, _, structure, a, *_ in book["Metals"].iter_rows(min_row=3, values_only=True)
          if s and isinstance(a, (int, float)) and re.fullmatch(r"[A-Z][a-z]?(3[A-Z][a-z]?)?", s)}
book.close()

assert MXENE in mxenes, f"Choose one of {list(mxenes)}"
assert METAL in metals, f"Choose one of {list(metals)}"
a = mxenes[MXENE]
a_m, metal_structure = metals[METAL]
print(f"{MXENE}: a = {a} Å    {METAL} ({metal_structure}): a = {a_m} Å")

## MXene: conventional (orthogonal) cell

Layers bottom to top from the formula: the metal with count 2 is outside, C/N layers follow their order in the formula, O2 adds an O layer on each side.

Sites: AβC cycles A → B → C; AβA puts metals on one site and C/N on the next. O sits on the site free of its two nearest layers. The labels are then shifted so the bottom layer is on A.

In [ ]:
D_MX = 0.385 * a        # Å, M–X layer spacing
BOND_MO = 2.0           # Å, M–O bond that sets the O layer height

# layers, bottom to top
tokens = [(e, int(n or 1)) for e, n in re.findall(r"([A-Z][a-z]?)(\d*)", MXENE)]
ms = [(e, n) for e, n in tokens if e not in ("C", "N", "O")]
xs = [e for e, n in tokens if e in ("C", "N") for _ in range(n)]
outer = next((e for e, n in ms if n == 2 and len(ms) > 1), ms[0][0])
inner = [e for e, n in ms for _ in range(n) if e != outer] if len(ms) > 1 else [outer] * (len(xs) - 1)
tm_layers = [outer] + inner + [outer]
elements = [tm_layers[0]] + [e for pair in zip(xs, tm_layers[1:]) for e in pair]
if dict(tokens).get("O"):
    elements = ["O"] + elements + ["O"]

# stacking sites
core = [i for i, e in enumerate(elements) if e != "O"]
centre = (core[0] + core[-1]) // 2
if STACKING == "ABC":
    sites = ["ABC"[(i - centre) % 3] for i in range(len(elements))]
else:
    sites = ["B" if e in ("C", "N") else "A" for e in elements]
if elements[0] == "O":
    sites[0] = ({"A", "B", "C"} - {sites[1], sites[2]}).pop()
if elements[-1] == "O":
    sites[-1] = ({"A", "B", "C"} - {sites[-2], sites[-3]}).pop()
k = "ABC".index(sites[0])                 # shift labels so the bottom layer is on A
sites = ["ABC"[("ABC".index(s) - k) % 3] for s in sites]

# heights above the bottom layer
heights = [0.0]
for lower, upper in zip(elements, elements[1:]):
    heights.append(heights[-1] + (math.sqrt(BOND_MO**2 - a**2 / 3) if "O" in (lower, upper) else D_MX))

c = heights[-1] + VACUUM                                 # top layer sits exactly at z = c
lattice = Lattice.orthorhombic(a, np.sqrt(3) * a, c)     # a × √3a × c, all angles 90°

# hexagonal site A (0,0) → (0, 0); B (1/3,2/3) → (0, 1/3); C (2/3,1/3) → (1/2, 1/6)
SITES = {"A": (0, 0), "B": (0, 1 / 3), "C": (1 / 2, 1 / 6)}

species, coords = [], []
for s, site, h in zip(elements, sites, heights):
    x, y = SITES[site]
    for dx, dy in [(0, 0), (0.5, 0.5)]:                  # the rectangular cell holds two copies of each layer
        species.append(s)
        coords.append([(x + dx) % 1, (y + dy) % 1, (VACUUM + h) / c])

mxene = Structure(lattice, species, coords, to_unit_cell=False)
mxene.to(filename=f"POSCAR_{MXENE}")
print(" ".join(f"{e}({s})" for e, s in zip(elements, sites)))

## Metal: conventional cubic cell

In [ ]:
if metal_structure == "FCC":                              # metal at 4a, 4 atoms
    metal = Structure.from_spacegroup("Fm-3m", Lattice.cubic(a_m), [METAL], [[0, 0, 0]])
    number = 225
else:                                                     # L12 A3B: B at 1a, A at 3c
    A, B = re.fullmatch(r"([A-Z][a-z]?)3([A-Z][a-z]?)", METAL).groups()
    metal = Structure.from_spacegroup("Pm-3m", Lattice.cubic(a_m), [B, A], [[0, 0, 0], [0, 0.5, 0.5]])
    number = 221

sga = SpacegroupAnalyzer(metal, symprec=1e-3)
assert sga.get_space_group_number() == number

metal.to(filename=f"POSCAR_{METAL}")
print(metal.composition, sga.get_space_group_symbol())

## In-plane replicas giving (approximately) the same box

In [ ]:
replicas = []          # per direction: (MXene replicas, metal replicas, strain on MXene)
for L_mx, L_m in zip(mxene.lattice.abc[:2], metal.lattice.abc[:2]):
    for n in range(1, MAX_REPLICAS + 1):
        m = max(1, round(n * L_m / L_mx))
        strain = n * L_m / (m * L_mx) - 1
        if abs(strain) <= TOL:
            replicas.append((m, n, strain))
            break
    else:
        raise ValueError(f"No match within {TOL:.1%} up to {MAX_REPLICAS} replicas; raise TOL or MAX_REPLICAS")

(mx_x, m_x, eps_x), (mx_y, m_y, eps_y) = replicas
print(f"x: {mx_x} MXene × {mxene.lattice.a:.4f} Å  vs  {m_x} {METAL} × {metal.lattice.a:.4f} Å  ->  MXene strain {eps_x:+.2%}")
print(f"y: {mx_y} MXene × {mxene.lattice.b:.4f} Å  vs  {m_y} {METAL} × {metal.lattice.b:.4f} Å  ->  MXene strain {eps_y:+.2%}")

## Location of the metal

Stacking the cubic cell puts metal (001) on MXene (0001), with metal [100] along MXene x.

In [ ]:
metal_super = metal.copy()
metal_super.make_supercell([m_x, m_y, math.ceil(METAL_LAYERS / 2)])          # two (001) planes per cubic cell
metal_super.remove_sites([k for k, s in enumerate(metal_super) if s.coords[2] > (METAL_LAYERS - 1) * a_m / 2 + 1e-3])

box_x, box_y = m_x * metal.lattice.a, m_y * metal.lattice.b               # whole metal cells: the metal keeps its lattice
a_s, b_s = box_x / mx_x, box_y / mx_y                                       # strained MXene cell
z_top = c + GAP + metal_super.cart_coords[:, 2].max()                        # vacuum only below the MXene
interface_lattice = Lattice.orthorhombic(box_x, box_y, z_top)

species, coords = [], []
for i in range(mx_x):                                           # MXene replicas, copied by hand: make_supercell
    for j in range(mx_y):                                       # would wrap the top layer from z = c to z = 0
        for s in mxene:                                         # same fractional x, y in the strained cell
            species.append(s.species_string)
            coords.append([(i + s.frac_coords[0]) * a_s, (j + s.frac_coords[1]) * b_s, s.coords[2]])

for s in metal_super:                                           # unstrained metal
    species.append(s.species_string)
    x = (s.frac_coords[0] + SHIFT[0] / box_x) % 1 * box_x
    y = (s.frac_coords[1] + SHIFT[1] / box_y) % 1 * box_y
    coords.append([x, y, c + GAP + s.coords[2]])

interface = Structure(interface_lattice, species, coords, coords_are_cartesian=True, to_unit_cell=False)
interface.get_sorted_structure().to(filename="POSCAR")
print(interface.composition, f"box {box_x:.3f} × {box_y:.3f} × {z_top:.3f} Å")